# Why does the score depend on the hosting school? Other models and school-history features

Follow-up to `random-forest-xgboost.ipynb`. Same 2024 train / 2025 test split. Three questions:

1. Does a different model type help?
2. How much of the score is just "which school is hosting"?
3. If we add each school's **earlier attendance** as a pregame feature, what happens?

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

X_train = pd.read_csv("../data/X_train.csv"); X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze(); y_test = pd.read_csv("../data/y_test.csv").squeeze()

def make_pre(X):
    cat = X.select_dtypes(include=["object", "str", "bool", "category"]).columns.tolist()
    num = [c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                          ("e", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat)])
cv = StratifiedKFold(5, shuffle=True, random_state=42)

## 1. Other model types
Each model is tuned with 5-fold CV on the 2024 training data only, then scored once on 2025. Run with and without `homeTeam` / `awayTeam`.

In [2]:
cands = [
 ("Logistic regression, tuned C", lambda: LogisticRegression(max_iter=3000, random_state=42), {"clf__C": [0.03, 0.1, 0.3, 1, 3, 10]}),
 ("Logistic regression, L1", lambda: LogisticRegression(penalty="l1", solver="liblinear", random_state=42), {"clf__C": [0.1, 0.3, 1, 3, 10]}),
 ("HistGradientBoosting", lambda: HistGradientBoostingClassifier(random_state=42),
    {"clf__max_depth": [2, 3, None], "clf__learning_rate": [0.03, 0.08], "clf__max_iter": [150, 300]}),
 ("SVM (RBF)", lambda: SVC(probability=True, random_state=42), {"clf__C": [0.3, 1, 3, 10], "clf__gamma": ["scale", 0.01]}),
 ("Neural net (MLP)", lambda: MLPClassifier(max_iter=800, random_state=42, early_stopping=True),
    {"clf__hidden_layer_sizes": [(32,), (64, 32)], "clf__alpha": [1e-3, 1e-1]}),
]
rows = []
for drop in ([], ["homeTeam", "awayTeam"]):
    a, b = X_train.drop(columns=drop), X_test.drop(columns=drop)
    for name, mk, grid in cands:
        best = GridSearchCV(Pipeline([("pre", make_pre(a)), ("clf", mk())]), grid, scoring="f1", cv=cv).fit(a, y_train).best_estimator_
        p = best.predict_proba(b)[:, 1]
        rows.append({"model": name, "team names": "no" if drop else "yes",
                     "F1": f1_score(y_test, p >= .5), "ROC-AUC": roc_auc_score(y_test, p)})
pd.DataFrame(rows).round(3)

,model,team names,F1,ROC-AUC
0,"Logistic regression, tuned C",yes,0.799,0.900
1,"Logistic regression, L1",yes,0.809,0.906
2,HistGradientBoosting,yes,0.753,0.872
3,SVM (RBF),yes,0.774,0.890
4,Neural net (MLP),yes,0.749,0.874
5,"Logistic regression, tuned C",no,0.730,0.839
6,"Logistic regression, L1",no,0.730,0.837
7,HistGradientBoosting,no,0.753,0.872
8,SVM (RBF),no,0.726,0.834
9,Neural net (MLP),no,0.724,0.826


**Reading this:** with team names, the L1 logistic regression is best (F1 about 0.81, ROC-AUC 0.91), a small step above the baseline. Without team names every model falls to about 0.73 to 0.75. Tree models, SVM and the neural net do not help on 815 training games.

## 2. How much is "which school is hosting"?

In [3]:
feat = pd.read_csv("../data/features.csv")
clean = pd.read_csv("../data/full_dataset_clean.csv")[["id", "startDate", "attendance_pct"]]
d = feat.merge(clean, on="id", how="left"); d["startDate"] = pd.to_datetime(d["startDate"], utc=True)
tr, te = d[d.season == 2024], d[d.season == 2025].copy()

school_mean_24 = tr.groupby("homeTeam")["attendance_pct"].mean()
print("corr(school's 2024 mean attendance %, game's 2025 attendance %):",
      round(te["attendance_pct"].corr(te["homeTeam"].map(school_mean_24)), 3))

rate24 = tr.groupby("homeTeam")["high_attendance"].mean()
te["rate24"] = te["homeTeam"].map(rate24)
stable = te["rate24"].isin([0, 1]); mixed = te["rate24"].between(0.01, 0.99)
print("2025 test games from schools that were all-high or all-low in 2024:", int(stable.sum()), "| mixed schools:", int(mixed.sum()))
for name, m in [("stable schools", stable), ("mixed schools", mixed)]:
    print(f"school-lookup accuracy on {name}: {accuracy_score(te.high_attendance[m], (te.rate24[m] >= .5).astype(int)):.3f}")

corr(school's 2024 mean attendance %, game's 2025 attendance %): 0.818
2025 test games from schools that were all-high or all-low in 2024: 544 | mixed schools: 285
school-lookup accuracy on stable schools: 0.903
school-lookup accuracy on mixed schools: 0.691


**Reading this:** a school's 2024 average attendance % correlates about 0.82 with its 2025 games. About two thirds of test games come from schools that were always high or always low in 2024, and a lookup gets those right about 90% of the time. The hard cases are the schools in between.

In [4]:
# Does the model rank games *within* a school? (AUC inside each school, averaged by number of games)
def within_school_auc(df, score):
    out, w = [], []
    for _, g in df.assign(s=score).groupby("homeTeam"):
        if g.high_attendance.nunique() == 2 and len(g) >= 4:
            out.append(roc_auc_score(g.high_attendance, g.s)); w.append(len(g))
    return np.average(out, weights=w), len(out)

te25 = d[d.season == 2025].reset_index(drop=True)
rows = []
for name, mk, drop in [("Logistic regression, with team names", lambda: LogisticRegression(max_iter=1000, random_state=42), []),
                       ("Logistic regression, no team names", lambda: LogisticRegression(max_iter=1000, random_state=42), ["homeTeam", "awayTeam"]),
                       ("Random Forest, no team names", lambda: RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1), ["homeTeam", "awayTeam"]),
                       ("XGBoost, no team names", lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric="logloss"), ["homeTeam", "awayTeam"])]:
    a, b = X_train.drop(columns=drop), X_test.drop(columns=drop)
    p = Pipeline([("pre", make_pre(a)), ("clf", mk())]).fit(a, y_train).predict_proba(b)[:, 1]
    w, n = within_school_auc(te25, p)
    rows.append({"model": name, "overall ROC-AUC": roc_auc_score(y_test, p), "within-school ROC-AUC": w, "schools used": n})
pd.DataFrame(rows).round(3)

,model,overall ROC-AUC,within-school ROC-AUC,schools used
0,"Logistic regression, with team names",0.893,0.773,56
1,"Logistic regression, no team names",0.838,0.755,56
2,"Random Forest, no team names",0.870,0.757,56
3,"XGBoost, no team names",0.873,0.751,56


**Reading this:** inside a single school, the game-level features still rank high vs low games well above chance (about 0.75, where 0.5 is a coin flip). So the model is not only memorizing schools. Knowing the school adds a big chunk on top (overall ROC-AUC 0.84 to 0.89).

## 3. School-history features (all known before kickoff)
For each home game we add the school's earlier attendance: average attendance % over all its earlier labeled home games, the last game, the average of the last three, and the earlier share of high games. 2024 week-1 games have no earlier game, so those are blank (imputed with the median in the pipeline).

In [5]:
d = d.sort_values(["homeTeam", "startDate"]).reset_index(drop=True)
g = d.groupby("homeTeam")["attendance_pct"]
d["prior_mean_pct"] = g.transform(lambda s: s.shift(1).expanding().mean())
d["last_pct"] = g.shift(1)
d["last3_pct"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
d["prior_high_rate"] = d.groupby("homeTeam")["high_attendance"].transform(lambda s: s.shift(1).expanding().mean())
d["n_prior_games"] = g.cumcount()
H = ["prior_mean_pct", "last_pct", "last3_pct", "prior_high_rate", "n_prior_games"]
d = d.sort_values("id").reset_index(drop=True)
dtr, dte = d[d.season == 2024], d[d.season == 2025]
print("share of rows with no earlier game: train", dtr["prior_mean_pct"].isna().mean().round(3), "| test", dte["prior_mean_pct"].isna().mean().round(3))

base = [c for c in X_train.columns if c not in ["homeTeam", "awayTeam"]]
P4 = ["SEC", "Big Ten", "Big 12", "ACC"]
mk = {"Logistic regression": lambda: LogisticRegression(max_iter=1000, random_state=42),
      "Random Forest": lambda: RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
      "XGBoost": lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric="logloss")}
rows = []
for name, f in mk.items():
    for label, cols in [("no history", base), ("+ school history", base + H)]:
        pipe = Pipeline([("pre", make_pre(dtr[cols])), ("clf", f())]).fit(dtr[cols], dtr.high_attendance)
        p = pipe.predict_proba(dte[cols])[:, 1]; pred = (p >= .5).astype(int); y = dte.high_attendance.values
        m = dte.homeConference.isin(P4).values
        rows.append({"model": name, "features": label, "F1": f1_score(y, pred), "ROC-AUC": roc_auc_score(y, p),
                     "within-school AUC": within_school_auc(dte.reset_index(drop=True), p)[0],
                     "recall Power 4": recall_score(y[m], pred[m]), "recall other": recall_score(y[~m], pred[~m])})
pd.DataFrame(rows).round(3)

share of rows with no earlier game: train 0.161 | test 0.006


,model,features,F1,ROC-AUC,within-school AUC,recall Power 4,recall other
0,Logistic regression,no history,0.734,0.838,0.755,0.847,0.312
1,Logistic regression,+ school history,0.793,0.919,0.739,0.824,0.338
2,Random Forest,no history,0.747,0.870,0.748,0.844,0.262
3,Random Forest,+ school history,0.819,0.923,0.527,0.905,0.412
4,XGBoost,no history,0.740,0.874,0.743,0.807,0.362
5,XGBoost,+ school history,0.796,0.921,0.636,0.878,0.375


**Reading this (all numbers are from one split, untuned history models):**
- Adding school history lifts overall F1 from 0.73–0.75 to 0.79–0.82 and ROC-AUC to 0.92, so it is the way past 0.8.
- It does **not** improve ranking inside a school (within-school AUC stays flat for logistic regression and drops for the tree models). The gain is the model knowing each school's usual crowd more precisely, not better understanding of what moves a single game.
- Recall outside the Power 4 improves a little (0.26 to 0.31 before, about 0.34 to 0.41 after) but stays well below Power 4 recall.

So the headline score and "understanding what drives attendance" are separate goals. Report both: overall F1 and within-school AUC.